In [4]:
import pandas as pd


Evento, fecha de apertura y días a usar: lunes a viernes de tres semanas antes y tres después, sin festivos ni días atípicos.
Juntamos los archivos diarios de un evento en una sola tabla, con una fila por estación, día y hora.

In [5]:
evento = "calle72"
fecha_evento = pd.to_datetime("2026-08-22")
carpeta = "../data/raw/validaciones/calle72/"

fechas = [
    "20260803","20260804","20260805","20260806",
    "20260810","20260811","20260812",
    "20260813","20260814","20260818","20260819",
    "20260820","20260821","20260824","20260825","20260826",
    "20260827","20260828","20260831","20260901",
    "20260902","20260903","20260904","20260907","20260908",
    "20260909","20260910","20260911"
]

In [6]:
resumenes = []

for fecha in fechas:
    df = pd.read_csv(carpeta + "validacionTroncal" + fecha + ".zip",
                     usecols=["Fecha_Transaccion","Estacion_Parada","Linea"])

    df["Fecha_Transaccion"] = pd.to_datetime(df["Fecha_Transaccion"])
    df["fecha"] = df["Fecha_Transaccion"].dt.normalize()
    df["hora"] = df["Fecha_Transaccion"].dt.hour
    df = df[df["fecha"] == pd.to_datetime(fecha)]

    df["codigo"] = df["Estacion_Parada"].str.strip().str[1:6]

    resumen = (df.groupby(["fecha","codigo","hora"])
                 .agg(validaciones=("codigo","count"),
                      estacion=("Estacion_Parada","first"),
                      linea=("Linea","first"))
                 .reset_index())
    resumenes.append(resumen)

tabla = pd.concat(resumenes, ignore_index=True)
tabla.shape

(81191, 6)


Dejamos un solo nombre por código de estación y agregamos si el día es antes o después del evento.

In [7]:
nombres = tabla.groupby("codigo")["estacion"].first()
lineas = tabla.groupby("codigo")["linea"].first()
tabla["estacion"] = tabla["codigo"].map(nombres).str.strip().str[8:]
tabla["linea"] = tabla["codigo"].map(lineas).str.strip()

tabla["evento"] = evento
tabla["periodo"] = "antes"
tabla.loc[tabla["fecha"] >= fecha_evento,"periodo"] = "después"

dias = ["lunes","martes","miércoles","jueves","viernes","sábado","domingo"]
tabla["dia_semana"] = tabla["fecha"].dt.dayofweek.map(lambda d: dias[d])

tabla.head()

,fecha,codigo,hora,validaciones,estacion,linea,evento,periodo,dia_semana
0,2026-08-03,02000,4,584,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
1,2026-08-03,02000,5,4584,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
2,2026-08-03,02000,6,9262,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
3,2026-08-03,02000,7,9362,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
4,2026-08-03,02000,8,6139,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes


In [8]:
tabla["fecha"].nunique(), tabla.duplicated(["fecha","codigo","hora"]).sum()

(28, np.int64(0))

In [9]:
tabla.to_csv("../data/processed/validaciones_" + evento + ".csv", index=False)

Notas

Calle 53 aparece en los datos como Marly - Calle 57 (código 09131). No usamos Hora_Pico_SN porque marca Peak Time en todas las filas.